# 12 – Performance tuning: broadcast ve shuffle join

Amaç: aynı join sorgusunu iki stratejiyle (`BROADCAST`, `SHUFFLE_MERGE`) çalıştırıp join stratejisinin ve skew'ün etkisini ayrı ayrı ölçmek.

Ölçüm: `EXPLAIN` + süre (`collect`, 3 run, en kısa süre `min_s`) + Query Profile.

Tablolar: `workspace.perf.trans_big` ve `workspace.perf.trans_skewed` (factor 100, 105,6M satır; skewed tabloda hot account 8261: 31,4M satır).

Alttaki hücre sorguyu ve ölçüm function'ını tanımlar, `SHUFFLE_MERGE` planını gösterir.

In [0]:
import time

PERF_SCHEMA = "workspace.perf"
TABLES = ["trans_big", "trans_skewed"]

# Aynı sorgu: sadece join hint'i değişir (BROADCAST veya SHUFFLE_MERGE)
def join_sql(table_name, join_hint):
    sql_text = f"""
        SELECT /*+ {join_hint}(a) */
               a.district_id,
               COUNT(*) AS trans_count,
               SUM(t.amount) AS total_amount
        FROM {PERF_SCHEMA}.{table_name} t
        JOIN workspace.silver.account a
          ON t.account_id = a.account_id
        GROUP BY a.district_id
    """
    return sql_text

# 3 run, en kısa süre; ilk run'daki başlatma maliyeti min ile elenir
def measure(sql_text, runs=3):
    durations = []
    for run_no in range(runs):
        start = time.time()
        spark.sql(sql_text).collect()
        durations.append(time.time() - start)
    return round(min(durations), 2)

spark.sql(join_sql("trans_skewed", "SHUFFLE_MERGE")).explain()

## Broadcast ve shuffle join

Aynı join sorgusu 2 hint × 2 tablo ile ölçülür (4 ölçüm, ~1–3 dk).

Okuma: aynı hint'te `trans_big` ↔ `trans_skewed` farkı **skew etkisi**; aynı tabloda `BROADCAST` ↔ `SHUFFLE_MERGE` farkı **join stratejisi etkisi**.

Not: `SortMergeJoin` Photon'da desteklenmiyor; shuffle join row-based çalışır.

In [0]:
HINTS = ["BROADCAST", "SHUFFLE_MERGE"]

results = []
for join_hint in HINTS:
    for table_name in TABLES:
        min_s = measure(join_sql(table_name, join_hint))
        results.append((join_hint, table_name, min_s))
        print(join_hint, table_name, min_s)

df_join_results = spark.createDataFrame(results, ["join_hint", "table_name", "min_s"])
display(df_join_results)

## Sonuç ve Query Profile bulgusu

| Hint | `trans_big` (sn) | `trans_skewed` (sn) |
|---|---|---|
| `BROADCAST` | 1,24 | 1,21 |
| `SHUFFLE_MERGE` | 6,17 | 6,40 |

Query Profile (`SHUFFLE_MERGE` + `trans_skewed`): wall-clock 6,1 sn, toplam task süresi 32,5 sn (join ~28 sn, shuffle 5,1 sn, sort 2,7 sn), Photon %26, peak memory 1,25 GB, spill 0.

Karşılaştırma (`BROADCAST` + `trans_big`): wall-clock 1,1 sn, toplam task süresi 1,2 sn, Photon %97; büyük tarafta shuffle yok.

**Join stratejisi:** Broadcast join ~5 kat hızlı. Shuffle join'de 105,6M satır `account_id`'ye göre shuffle ve sort ediliyor; Photon `SortMergeJoin`'i desteklemediği için join row-based çalışıyor.

**Skew:** Aynı hint'te `trans_big` ↔ `trans_skewed` farkı sadece ~%4. Profilde skew metriği yok; AQE skew join gözlenmedi. Shuffle join'in maliyetini skew değil, shuffle + sort + row-based join belirliyor.